In [9]:
import sys
import os

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)


In [1]:
import json
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

# reuse pipelines from Phase 4


In [2]:
df = pd.read_csv("../data/titanic.csv")

target_col = "Survived"
X = df.drop(columns=[target_col])
y = df[target_col]


In [3]:
with open("../outputs/preprocessing_plan.json") as f:
    preprocessing_plan = json.load(f)

X = X.drop(columns=preprocessing_plan["drop_columns"])


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [5]:
baseline_model = LogisticRegression(
    max_iter=1000,
    solver="lbfgs"
)


In [7]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler,
    OneHotEncoder, PowerTransformer
)
from sklearn.impute import SimpleImputer


In [8]:
import numpy as np

numerical_cols = X.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X.select_dtypes(exclude=np.number).columns.tolist()


In [10]:
from src.pipelines import build_pipelines

pipelines = build_pipelines(X_train, preprocessing_plan)


In [11]:
from sklearn.pipeline import Pipeline

evaluated_pipelines = {}

for name, pipe in pipelines.items():
    evaluated_pipelines[name] = Pipeline(steps=[
        ("preprocessor", pipe.named_steps["preprocessor"]),
        ("model", baseline_model)
    ])


In [12]:
results = []

for name, pipe in evaluated_pipelines.items():
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    results.append({
        "pipeline": name,
        "accuracy": round(acc, 4),
        "f1_score": round(f1, 4)
    })


In [13]:
results_df = pd.DataFrame(results).sort_values(
    by="f1_score", ascending=False
)

results_df


,pipeline,accuracy,f1_score
1,minmax,0.8101,0.7344
0,standard,0.7877,0.7077
2,robust,0.7877,0.7077


In [17]:
best_pipeline = results_df.iloc[0]["pipeline"]
best_pipeline


'minmax'

In [18]:
results_df.to_csv("../outputs/pipeline_evaluation.csv", index=False)

with open("../outputs/best_pipeline.json", "w") as f:
    json.dump({"best_pipeline": best_pipeline}, f, indent=4)


finals tuff


In [25]:
num_cols = []
cat_cols = []

for col, rules in preprocessing_plan.items():
    if col not in df.columns:
        continue  # safety guard

    if "encoding" in rules:
        cat_cols.append(col)
    else:
        num_cols.append(col)


In [26]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
import numpy as np
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("log", FunctionTransformer(np.log1p, validate=False)),
    ("scaler", StandardScaler())
])


In [27]:
from sklearn.preprocessing import OneHotEncoder
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])


In [28]:
from sklearn.compose import ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, num_cols),
        ("cat", categorical_pipeline, cat_cols)
    ]
)


In [29]:
X = df[num_cols + cat_cols]

X_processed = preprocessor.fit_transform(X)


In [30]:
feature_names_num = num_cols
feature_names_cat = preprocessor.named_transformers_[
    "cat"
].named_steps["encoder"].get_feature_names_out(cat_cols)

final_feature_names = list(feature_names_num) + list(feature_names_cat)


NotFittedError: This OneHotEncoder instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.